# Unsupervised Classical ML + Recommenders — Masterclass

The second notebook in the **classical ML algorithms series**. Goal: when an interviewer asks "explain k-means and write it from scratch," "what's the difference between PCA and t-SNE," or "build me a basic collaborative filtering recommender" — you have the code, the math, and the interview-line answer ready.

## What's covered (in execution order)

**Clustering**
1. **K-Means** — Lloyd's algorithm from scratch + k-means++ + sklearn + k selection (elbow + silhouette)
2. **DBSCAN** — density-based clustering, the eps/min_samples grid, when DBSCAN beats k-means
3. **Hierarchical clustering** — agglomerative + dendrogram visualization + linkage criteria
4. **Gaussian Mixture Models** — EM from scratch on a 2-cluster toy + sklearn for real use

**Dimensionality reduction**
5. **PCA** — from scratch via eigendecomposition AND via SVD + sklearn + when PCA hurts
6. **t-SNE** — conceptual math (perplexity, KL divergence) + sklearn + MNIST visualization
7. **UMAP** — comparison with t-SNE on the same data

**Text representations**
8. **TF-IDF** — from scratch (TF × IDF) + sklearn's `TfidfVectorizer`

**Recommender systems**
9. **Collaborative filtering** — user-user, item-item, matrix factorization (SVD-style) from scratch with SGD

**Anomaly detection (bonus)**
10. **Isolation Forest** — brief intro for outlier detection

## Per-algorithm template
Same 7-element structure as Notebook 1:
1. **One-paragraph story** — mental model, when to reach for it
2. **Core math** — the objective / update rule / loss
3. **From-scratch NumPy implementation** (where reasonable)
4. **sklearn implementation** with hyperparameter walk-through
5. **Visualization** — cluster assignment, projection, etc.
6. **Cheat table** — time complexity, assumptions, drawbacks, needs scaling, interpretability
7. **Interview Q&A** with at least one trap question

## What's explicitly NOT from-scratch (and why)
- **t-SNE, UMAP** — full from-scratch involves perplexity calibration + non-convex KL minimization. Pedagogically pointless; never asked in interviews. Conceptual mechanics + sklearn only.
- **DBSCAN** — interview-tested for *understanding* (when, why) not implementation. Sklearn only.
- **Hierarchical clustering** — same. Sklearn + dendrogram visual only.

## Stack (May 2026)
- Python 3.11+, pandas 3.0.x, numpy 2.4.x
- scikit-learn 1.8.x
- scipy (for dendrogram)
- umap-learn 0.5.x


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility.
RNG = np.random.default_rng(seed=42)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
pd.set_option("display.precision", 4)

sns.set_theme(style="whitegrid", context="notebook")

import sklearn
print(f"numpy {np.__version__}  |  pandas {pd.__version__}  |  sklearn {sklearn.__version__}")
import umap
print(f"umap-learn {umap.__version__}")


---
# 1. K-Means

The default clustering algorithm. Partitions $n$ points into $k$ clusters by minimizing the **within-cluster sum of squares (WCSS)**. Lloyd's algorithm — the iterative version everyone uses — is the canonical "implement this in 15 minutes" interview question for unsupervised ML.

## 1.1 Story

Given data $\{\mathbf{x}_1, \dots, \mathbf{x}_n\} \in \mathbb{R}^d$ and a number $k$, partition the data into $k$ clusters $C_1, \dots, C_k$ such that the sum of squared distances from each point to its assigned cluster center $\boldsymbol{\mu}_c$ is minimized:

$$\min_{C_1, \dots, C_k, \boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_k} \; \sum_{c=1}^k \sum_{\mathbf{x} \in C_c} \|\mathbf{x} - \boldsymbol{\mu}_c\|^2$$

This is NP-hard in general. Lloyd's algorithm is a greedy heuristic that finds a **local minimum**.

## 1.2 Lloyd's algorithm

```
Initialize k centroids (randomly, or via k-means++)
Repeat until convergence:
    1. Assignment step: assign each point to the nearest centroid
    2. Update step: recompute each centroid as the mean of points assigned to it
```

Each iteration is guaranteed to decrease (or hold) WCSS. Convergence is guaranteed in finite steps, but the result depends on initialization. **Run multiple times with different seeds and pick the lowest WCSS.**

## 1.3 k-means++ initialization (the standard improvement)

Random initialization can place two centroids very close together → poor local minimum. k-means++ fixes this:

```
1. Pick first centroid uniformly at random from data.
2. For each remaining point x, compute D(x) = distance to nearest existing centroid.
3. Pick next centroid with probability ∝ D(x)^2 (favors points far from existing centroids).
4. Repeat until k centroids are chosen.
```

Spreads initial centroids out → better-quality local minimum on average. sklearn uses k-means++ by default.

## 1.4 From-scratch implementation

In [ ]:
class KMeansScratch:
    '''
    K-Means via Lloyd's algorithm with k-means++ initialization.

    Time complexity:
      fit:     O(n * k * d * n_iter)
      predict: O(n * k * d)
    '''
    def __init__(self, k=3, max_iter=300, tol=1e-4, init="k-means++", random_state=0):
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.init = init
        self.random_state = random_state

    def _init_centroids(self, X):
        rng = np.random.default_rng(self.random_state)
        n, d = X.shape
        if self.init == "random":
            idx = rng.choice(n, size=self.k, replace=False)
            return X[idx].copy()

        # k-means++ initialization.
        centroids = np.empty((self.k, d))
        # Pick first centroid uniformly.
        centroids[0] = X[rng.integers(n)]
        # Squared distance to nearest existing centroid.
        d2 = np.sum((X - centroids[0])**2, axis=1)
        for c in range(1, self.k):
            # Probabilistic pick proportional to squared distance.
            probs = d2 / d2.sum()
            new_idx = rng.choice(n, p=probs)
            centroids[c] = X[new_idx]
            # Update distances — keep the min across existing centroids.
            new_d2 = np.sum((X - centroids[c])**2, axis=1)
            d2 = np.minimum(d2, new_d2)
        return centroids

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        centroids = self._init_centroids(X)
        self.history_ = []

        for it in range(self.max_iter):
            # 1. Assignment: each point to nearest centroid.
            # Vectorized squared Euclidean: (n, k) distance matrix.
            dists = ((X[:, None, :] - centroids[None, :, :])**2).sum(axis=2)
            labels = dists.argmin(axis=1)

            # WCSS (the loss).
            wcss = dists[np.arange(len(X)), labels].sum()
            self.history_.append(wcss)

            # 2. Update: centroid = mean of assigned points.
            new_centroids = np.array([
                X[labels == c].mean(axis=0) if (labels == c).any() else centroids[c]
                for c in range(self.k)
            ])

            # Convergence check: centroid shift below tolerance.
            shift = np.linalg.norm(new_centroids - centroids)
            centroids = new_centroids
            if shift < self.tol:
                break

        self.centroids_ = centroids
        self.labels_ = labels
        self.inertia_ = self.history_[-1]
        self.n_iter_ = it + 1
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        dists = ((X[:, None, :] - self.centroids_[None, :, :])**2).sum(axis=2)
        return dists.argmin(axis=1)


# Validate against sklearn.
from sklearn.cluster import KMeans as SKKMeans
from sklearn.datasets import make_blobs

X_km, y_km = make_blobs(n_samples=500, centers=4, cluster_std=0.8,
                         random_state=42, n_features=2)

mine = KMeansScratch(k=4, random_state=0).fit(X_km)
theirs = SKKMeans(n_clusters=4, n_init=10, random_state=0).fit(X_km)

print(f"My WCSS (inertia):       {mine.inertia_:.2f}")
print(f"sklearn WCSS (inertia):  {theirs.inertia_:.2f}")
print(f"My n_iter:               {mine.n_iter_}")

# Cluster labels may be permuted (cluster 0 in mine != cluster 0 in sklearn) — that's fine.
# Check agreement via Adjusted Rand Index (invariant to label permutation).
from sklearn.metrics import adjusted_rand_score
print(f"Cluster assignment agreement (ARI): {adjusted_rand_score(mine.labels_, theirs.labels_):.3f}")


## 1.5 Visualizing convergence

In [ ]:
# Run from scratch and show the WCSS history + final clusters.
mine = KMeansScratch(k=4, random_state=0).fit(X_km)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: WCSS over iterations.
axes[0].plot(mine.history_, marker="o")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("WCSS (inertia)")
axes[0].set_title(f"Lloyd's algorithm — converges in {mine.n_iter_} iterations")

# Right: final clusters.
axes[1].scatter(X_km[:, 0], X_km[:, 1], c=mine.labels_, cmap="tab10", s=25, edgecolors="black")
axes[1].scatter(mine.centroids_[:, 0], mine.centroids_[:, 1],
                 c="red", marker="X", s=200, edgecolors="black", linewidth=2,
                 label="Centroids")
axes[1].set_title("Final clustering")
axes[1].legend()
plt.tight_layout()
plt.show()


## 1.6 Choosing k — the elbow method and silhouette score

The single hardest question with k-means: **how do you pick k?**

Two complementary heuristics:

**Elbow method.** Plot WCSS vs k. Pick the k where the curve bends sharply (the "elbow") — adding more clusters past that yields diminishing returns. Subjective, but cheap.

**Silhouette score.** For each point $i$:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
where $a(i)$ = mean intra-cluster distance and $b(i)$ = mean nearest-other-cluster distance. Range $[-1, 1]$: 1 = perfectly clustered, 0 = on the boundary, -1 = misassigned. Average across all points to score the clustering. **Pick k that maximizes mean silhouette.**

In [ ]:
from sklearn.metrics import silhouette_score

# Try k from 2 to 10.
ks = range(2, 11)
wcss_list, sil_list = [], []

for k in ks:
    km = SKKMeans(n_clusters=k, n_init=10, random_state=0).fit(X_km)
    wcss_list.append(km.inertia_)
    sil_list.append(silhouette_score(X_km, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(ks), wcss_list, marker="o", color="steelblue")
axes[0].set_xlabel("k")
axes[0].set_ylabel("WCSS")
axes[0].set_title("Elbow method — pick the 'kink'")
axes[0].axvline(4, color="red", linestyle="--", alpha=0.5, label="True k=4")
axes[0].legend()

axes[1].plot(list(ks), sil_list, marker="o", color="darkgreen")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette method — pick the max")
axes[1].axvline(4, color="red", linestyle="--", alpha=0.5, label="True k=4")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Best k by silhouette: {list(ks)[np.argmax(sil_list)]}")


**Both methods correctly identify k=4 here.** In real data they often disagree — elbow says one thing, silhouette another. The honest answer is: **k-means doesn't tell you the right k; domain knowledge does.** Use these as starting points, then validate clusters with domain experts.

## 1.7 Cheat table

In [ ]:
kmeans_cheat = pd.DataFrame([
    ["K-Means", "O(n · k · d · n_iter)", "O(n · k · d)",
     "Yes", "No", "Medium (centroids interpretable)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])

kmeans_assumptions = pd.DataFrame([
    ["Clusters are roughly spherical (isotropic)", "Lloyd's uses Euclidean distance, treats all directions equally",
     "If clusters are elongated, use GMM with full covariance or DBSCAN"],
    ["Clusters are roughly equal in size", "Variance term dominated by large clusters",
     "Use weighted k-means or GMM"],
    ["Number of clusters k is known", "Algorithm requires k as input",
     "Use elbow/silhouette/gap statistic to pick k"],
    ["Features are scaled comparably", "Distance computation is sensitive to feature scale",
     "Standardize features before fitting"],
], columns=["Assumption", "Why it matters", "Fix when violated"])

print("Cheat table:")
print(kmeans_cheat.to_string(index=False))
print("\nAssumptions:")
print(kmeans_assumptions.to_string(index=False))


## 1.8 Interview Q&A

**Q: Walk me through Lloyd's algorithm.**
> Initialize k centroids (k-means++ for better init). Then repeat until convergence: (1) **assignment step** — each point goes to its nearest centroid. (2) **update step** — each centroid becomes the mean of its assigned points. Convergence is when centroids stop moving (or move less than a tolerance). Each iteration is guaranteed to decrease WCSS, so convergence is monotonic. Result is a **local minimum** of WCSS, not global — depends on initialization.

**Q: Why is k-means initialization important?**
> The final clustering is a local minimum of a non-convex objective. Bad initialization (two centroids near each other, one far from any data) → bad local minimum → wrong clusters. k-means++ initialization picks centroids one at a time, with each new centroid favored to be far from existing ones. Combined with `n_init=10` (run 10 times, keep best) you reliably find good solutions.

**Q (trap): I ran k-means with k=4 twice with different seeds and got different cluster assignments. Is something broken?**
> No, that's expected. Lloyd's algorithm finds a local minimum that depends on initialization. With small noise or close-to-equally-good clusterings, different seeds converge to different solutions. The fix: use `n_init=10` (sklearn does this by default) and the algorithm runs 10 times, returning the best WCSS. If results still vary substantially, the clusters aren't well-separated in your data — that's a data problem, not an algorithm problem.

**Q: Why does k-means need feature scaling?**
> The WCSS objective is Euclidean distance squared. If one feature has range 0-1 and another 0-10000, the second dominates the distance entirely. Always standardize features (zero mean, unit variance) before k-means. This is the #1 k-means bug in production.

**Q (trap): K-means doesn't work on my data. The clusters are elongated, not spherical. What should I use?**
> Two options: (1) **GMM with full covariance matrices** — each cluster gets its own covariance, so elongated/rotated clusters are modeled correctly. (2) **DBSCAN** — density-based, doesn't assume any cluster shape, just connectivity. K-means' spherical-cluster assumption is fundamental — Lloyd's uses Euclidean distance, which weights all directions equally. There's no way to "fix" k-means for elongated clusters; switch algorithms.

**Q: What's the difference between k-means and k-medoids?**
> K-means: centroid = mean of assigned points. Doesn't have to be a data point. Sensitive to outliers (mean shifts toward extreme values). K-medoids: centroid = actual data point that minimizes total distance to other points in its cluster. More robust to outliers, slower (O(n²) per iteration), and works with non-Euclidean distance metrics. Use k-medoids when outliers exist or you need a non-Euclidean metric.

**Q: When would you NOT use k-means?**
> When (a) clusters aren't spherical, (b) cluster sizes are very imbalanced, (c) k is hard to pick / unknown, (d) clusters have varying density (DBSCAN's strength), (e) data has many categorical features (use k-modes), (f) clusters overlap heavily (use GMM for soft assignment).


---
# 2. DBSCAN (Density-Based Spatial Clustering)

K-means' main rival. Two big advantages: (1) no need to pre-specify the number of clusters, and (2) handles non-spherical clusters. Two big disadvantages: (1) struggles with clusters of varying density, and (2) requires tuning `eps` and `min_samples`.

## 2.1 Story

A cluster is a **dense region of points**. A point is **dense** if it has at least `min_samples` points within distance `eps`. Connect dense points transitively; the resulting connected components are the clusters. Points not in any cluster are labeled **noise / outliers**.

## 2.2 Core definitions

- **Core point:** has `min_samples` neighbors within `eps`. These form the cluster "interior."
- **Border point:** has fewer than `min_samples` neighbors within `eps`, but is within `eps` of a core point. Goes to whichever core point's cluster it touches.
- **Noise point:** neither core nor border. Labeled -1 in sklearn.

## 2.3 The algorithm

```
1. For each point, find all neighbors within eps.
2. Mark points with ≥ min_samples neighbors as core points.
3. Form clusters: each core point + all core points within eps + their border points.
   (Build connected components of the core-point graph.)
4. Remaining points = noise.
```

Time complexity: $O(n^2)$ naive, $O(n \log n)$ with KD-tree (sklearn default, helps when $d$ is low).

## 2.4 Why DBSCAN beats k-means on non-spherical data

In [ ]:
# Compare k-means and DBSCAN on two-moons.
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=300, noise=0.08, random_state=0)

km = SKKMeans(n_clusters=2, n_init=10, random_state=0).fit(X_moon)
db = DBSCAN(eps=0.2, min_samples=5).fit(X_moon)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap="tab10", s=25, edgecolors="black")
axes[0].set_title("Ground truth (two moons)")

axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=km.labels_, cmap="tab10", s=25, edgecolors="black")
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                 c="red", marker="X", s=200, edgecolors="black", linewidth=2)
axes[1].set_title(f"K-Means (ARI={adjusted_rand_score(y_moon, km.labels_):.3f})")

axes[2].scatter(X_moon[:, 0], X_moon[:, 1], c=db.labels_, cmap="tab10", s=25, edgecolors="black")
axes[2].set_title(f"DBSCAN (ARI={adjusted_rand_score(y_moon, db.labels_):.3f})")

plt.tight_layout()
plt.show()


**The picture says it:** k-means cuts the moons in half (it can only produce linearly separated centroids). DBSCAN follows the density, recovering the moons cleanly. This is THE canonical example of when DBSCAN matters.

## 2.5 Choosing eps — the k-distance plot

The hardest part of DBSCAN: picking `eps`. Heuristic: plot the distance to the $k$-th nearest neighbor (where $k$ = `min_samples`) for each point, sorted. The "elbow" of this curve is a reasonable `eps`.

In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 5
nn = NearestNeighbors(n_neighbors=min_samples).fit(X_moon)
distances, _ = nn.kneighbors(X_moon)
# Distance to the k-th nearest neighbor (the furthest of the k nearest).
k_distances = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_distances)
ax.set_xlabel("Points (sorted by k-distance)")
ax.set_ylabel(f"Distance to {min_samples}-th nearest neighbor")
ax.set_title(f"k-distance plot for choosing eps (min_samples={min_samples})")
ax.axhline(0.2, color="red", linestyle="--", alpha=0.5, label="eps = 0.2")
ax.legend()
plt.show()

# The "elbow" (around y=0.2 here) is a good starting eps.


## 2.6 Cheat table

In [ ]:
dbscan_cheat = pd.DataFrame([
    ["DBSCAN", "O(n · log n) with KD-tree, O(n²) brute force",
     "Same as fit (no separate predict; use fit_predict or check core distances)",
     "Yes", "No", "Medium (labels but no cluster centers)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
dbscan_cheat


## 2.7 Interview Q&A

**Q: What's the main advantage of DBSCAN over k-means?**
> Three: (1) you don't have to pre-specify the number of clusters — DBSCAN discovers them from the data. (2) handles non-spherical clusters (the canonical two-moons / Swiss roll example). (3) explicitly labels noise points instead of forcing them into clusters.

**Q: What's the main disadvantage of DBSCAN?**
> DBSCAN assumes **constant density** across clusters. If one cluster is denser than another, no single `eps` works for both — small eps splits the loose cluster into pieces, large eps merges everything. HDBSCAN (hierarchical DBSCAN) is the modern fix: it tries multiple `eps` values and produces a hierarchy of clusters.

**Q (trap): My DBSCAN labeled 80% of points as noise. What went wrong?**
> Either (a) `eps` is too small — try increasing it; do a k-distance plot to pick a better value, or (b) `min_samples` is too high — try lowering it (default is 5; for high-dimensional data, set it to `2 * dim`). If after tuning, most points are still noise, your data doesn't have dense clusters in the chosen feature space — try dimensionality reduction first (PCA, UMAP) then DBSCAN on the lower-dimensional space.

**Q: How does DBSCAN handle high-dimensional data?**
> Poorly. Like all distance-based methods, DBSCAN suffers from the curse of dimensionality: as $d$ grows, all points become approximately equidistant, so `eps`-balls either contain all points or none. Rule of thumb: project to ≤10 dimensions with PCA/UMAP first if your data has many features. Some practitioners use cosine distance for text data, where the curse is less severe.

**Q (trap): DBSCAN didn't find any clusters in my data. It assigned everything to noise. But k-means with k=3 found 3 clusters. Which should I trust?**
> Be skeptical of k-means. K-means **always** produces k clusters — even on pure noise. DBSCAN says "I don't see density structure" and labels everything noise; this is genuine information. Plot the data, look at the k-distance plot, and ask whether the "clusters" k-means found are meaningfully different in distance from random partitions. The most common diagnostic is the **silhouette score** — if k-means' silhouette is near zero, the clusters are real arbitrary, not real.


---
# 3. Hierarchical Clustering (Agglomerative)

Builds a **tree of clusters** rather than a flat partition. Two strategies: **agglomerative** (bottom-up: start with each point as its own cluster, merge the closest pair iteratively) and **divisive** (top-down: start with everyone in one cluster, recursively split). Agglomerative is the default.

The output isn't a single clustering — it's a **dendrogram** that you cut at a chosen height to get any number of clusters.

## 3.1 The algorithm

```
1. Each point is its own cluster.
2. Compute pairwise distances between all clusters.
3. Repeat until only one cluster remains:
    a. Merge the two closest clusters.
    b. Update distances between the new merged cluster and all others.
4. Output: dendrogram tracking the merge sequence and heights.
```

Time complexity: $O(n^3)$ naive, $O(n^2 \log n)$ with better data structures. **Doesn't scale past ~10K samples.**

## 3.2 Linkage criteria — how do you define "closest pair of clusters"?

| Linkage | Distance between cluster $A$ and $B$ |
|---|---|
| **Single** | $\min_{a \in A, b \in B} d(a, b)$ — closest pair |
| **Complete** | $\max_{a \in A, b \in B} d(a, b)$ — farthest pair |
| **Average** | $\text{mean}_{a, b} d(a, b)$ — average pairwise |
| **Ward** | Increase in within-cluster variance from merging |

Behavior:
- **Single** linkage produces long, chaining clusters (sensitive to noise).
- **Complete** produces tight, compact clusters.
- **Average** is a compromise.
- **Ward** (the default in sklearn for Euclidean) minimizes variance — produces compact, similarly-sized clusters. Closest analog to k-means.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Generate a small dataset (hierarchical clustering is expensive — keep n small).
X_h, y_h = make_blobs(n_samples=50, centers=3, cluster_std=1.0, random_state=42)

# Compute the linkage matrix for the dendrogram.
Z = linkage(X_h, method="ward")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: dendrogram.
dendrogram(Z, ax=axes[0], color_threshold=10)
axes[0].set_title("Dendrogram (Ward linkage)")
axes[0].set_xlabel("Point index")
axes[0].set_ylabel("Distance (within-cluster variance increase)")
axes[0].axhline(10, color="red", linestyle="--", alpha=0.5, label="Cut → 3 clusters")
axes[0].legend()

# Right: clustering at 3 clusters.
hc = AgglomerativeClustering(n_clusters=3, linkage="ward").fit(X_h)
axes[1].scatter(X_h[:, 0], X_h[:, 1], c=hc.labels_, cmap="tab10", s=40, edgecolors="black")
axes[1].set_title(f"Cut at 3 clusters (ARI={adjusted_rand_score(y_h, hc.labels_):.3f})")

plt.tight_layout()
plt.show()


## 3.3 Comparison of linkages on the same data

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, method in zip(axes, ["single", "complete", "average", "ward"]):
    hc = AgglomerativeClustering(n_clusters=3, linkage=method).fit(X_h)
    ax.scatter(X_h[:, 0], X_h[:, 1], c=hc.labels_, cmap="tab10", s=40, edgecolors="black")
    ari = adjusted_rand_score(y_h, hc.labels_)
    ax.set_title(f"{method}-linkage\nARI={ari:.3f}")
plt.tight_layout()
plt.show()


## 3.4 Cheat table

In [ ]:
hier_cheat = pd.DataFrame([
    ["Hierarchical (agglomerative)", "O(n² log n) typical, O(n³) naive", "O(n²) memory",
     "Yes", "No", "High (full dendrogram is the model)"],
], columns=["Algorithm", "Train time", "Predict time / memory", "Needs scaling?", "Handles missing?", "Interpretability"])
hier_cheat


## 3.5 Interview Q&A

**Q: When would you use hierarchical clustering over k-means?**
> When (a) you want to **explore** cluster structure without committing to a `k` upfront — the dendrogram lets you cut at any level. (b) Your data is small (< 10K) so the $O(n^2)$ memory cost is fine. (c) You need a hierarchical interpretation — e.g., taxonomies, customer segments at multiple granularities. K-means is better for large datasets and when you have a target `k`.

**Q: What linkage criterion would you choose by default?**
> **Ward** for Euclidean distance — it produces compact, similarly-sized clusters and is the closest hierarchical analog to k-means. Use complete linkage if you want tight clusters and don't mind throwing out outliers. Use average linkage as a compromise. **Avoid single linkage unless you specifically want chaining** (e.g., for detecting filament structures in physics data).

**Q (trap): I ran hierarchical clustering on 100K points and it ran out of memory. Why?**
> The distance matrix is $O(n^2)$ = 10 billion entries for 100K points → ~80 GB at float64. Hierarchical clustering doesn't scale past ~10K samples. Either (a) subsample your data, (b) cluster on a low-dimensional embedding (PCA → 50 dims → cluster), or (c) use **BIRCH** (a streaming hierarchical method) or just switch to k-means or HDBSCAN.

**Q: What's the difference between agglomerative and divisive clustering?**
> Agglomerative: bottom-up. Start with each point as its own cluster, repeatedly merge the closest two. Standard sklearn implementation. Divisive: top-down. Start with everyone in one cluster, recursively split. Less common in practice; usually framed as repeated 2-means or spectral cuts.


---
# 4. Gaussian Mixture Models (GMM)

The "soft k-means." Model each cluster as a multivariate Gaussian; the dataset is a **mixture** of these Gaussians. Each point gets a **probability** of belonging to each cluster, not a hard assignment. Fit via Expectation-Maximization (EM).

GMM is strictly more flexible than k-means: it can model **elliptical** clusters (via full covariance matrices) and gives **probabilistic** cluster assignments.

## 4.1 The model

Density of a single point:
$$p(\mathbf{x}) = \sum_{c=1}^k \pi_c \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_c, \boldsymbol{\Sigma}_c)$$

Parameters per cluster:
- $\pi_c$ — mixing weight (prior probability of cluster $c$); $\sum_c \pi_c = 1$.
- $\boldsymbol{\mu}_c$ — cluster mean.
- $\boldsymbol{\Sigma}_c$ — cluster covariance (the "shape" of the cluster).

## 4.2 The EM algorithm

**E-step** (Expectation): compute responsibilities $\gamma_{ic}$ = probability that point $i$ belongs to cluster $c$:
$$\gamma_{ic} = \frac{\pi_c \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_c, \boldsymbol{\Sigma}_c)}{\sum_{c'} \pi_{c'} \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_{c'}, \boldsymbol{\Sigma}_{c'})}$$

**M-step** (Maximization): update parameters as weighted statistics:
$$N_c = \sum_i \gamma_{ic}, \quad \pi_c = N_c / n$$
$$\boldsymbol{\mu}_c = \frac{1}{N_c} \sum_i \gamma_{ic} \mathbf{x}_i$$
$$\boldsymbol{\Sigma}_c = \frac{1}{N_c} \sum_i \gamma_{ic} (\mathbf{x}_i - \boldsymbol{\mu}_c)(\mathbf{x}_i - \boldsymbol{\mu}_c)^\top$$

Iterate E and M until log-likelihood converges. Like Lloyd's, EM finds a **local maximum** — depends on initialization.

## 4.3 Covariance types

sklearn lets you constrain the covariance shape:

| `covariance_type` | $\boldsymbol{\Sigma}_c$ shape | Equivalent to |
|---|---|---|
| `spherical` | $\sigma_c^2 I$ (one variance per cluster) | "soft k-means" |
| `diag` | Diagonal | Independent features per cluster |
| `tied` | Same full $\boldsymbol{\Sigma}$ shared across clusters | LDA-like |
| `full` | Each cluster has its own full covariance | Maximum flexibility, most parameters |

## 4.4 From-scratch implementation

In [ ]:
class GMMScratch:
    '''
    Gaussian Mixture Model with full covariance, fit via EM.

    Time complexity:
      fit:     O(n · k · d² · n_iter)  (d² from inverting covariance matrices)
      predict: O(n · k · d²)
    '''
    def __init__(self, k=2, max_iter=100, tol=1e-4, random_state=0):
        self.k = k
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    def _multivariate_normal_logpdf(self, X, mu, Sigma):
        '''
        Compute log p(x | mu, Sigma) for each row of X.
        Uses log-determinant + Cholesky-style inversion for numerical stability.
        '''
        d = X.shape[1]
        diff = X - mu
        # Use pinv for robustness to near-singular covariance.
        Sigma_inv = np.linalg.pinv(Sigma)
        sign, logdet = np.linalg.slogdet(Sigma)
        if sign <= 0:
            logdet = -20.0  # fallback for numerical edge cases
        # Mahalanobis distance.
        mahal = np.einsum("ij,jk,ik->i", diff, Sigma_inv, diff)
        return -0.5 * (d * np.log(2 * np.pi) + logdet + mahal)

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        n, d = X.shape
        rng = np.random.default_rng(self.random_state)

        # Initialize: pick k random points as means, identity covariance, equal weights.
        idx = rng.choice(n, size=self.k, replace=False)
        self.means_ = X[idx].copy()
        self.covs_ = np.array([np.eye(d) for _ in range(self.k)])
        self.weights_ = np.ones(self.k) / self.k

        self.history_ = []

        for it in range(self.max_iter):
            # E-step: compute responsibilities (log space for numerical stability).
            log_resp = np.zeros((n, self.k))
            for c in range(self.k):
                log_resp[:, c] = np.log(self.weights_[c] + 1e-300) + \
                                  self._multivariate_normal_logpdf(X, self.means_[c], self.covs_[c])
            # Normalize: subtract max, exp, divide by sum (logsumexp trick).
            log_resp_max = log_resp.max(axis=1, keepdims=True)
            resp = np.exp(log_resp - log_resp_max)
            resp /= resp.sum(axis=1, keepdims=True)

            # Log-likelihood for convergence tracking.
            ll = np.sum(log_resp_max.flatten() + np.log(np.exp(log_resp - log_resp_max).sum(axis=1)))
            self.history_.append(ll)

            # M-step: update means, covs, weights.
            N_c = resp.sum(axis=0) + 1e-10           # (k,)
            self.weights_ = N_c / n
            for c in range(self.k):
                self.means_[c] = (resp[:, c:c+1] * X).sum(axis=0) / N_c[c]
                diff = X - self.means_[c]
                self.covs_[c] = (resp[:, c:c+1] * diff).T @ diff / N_c[c]
                # Regularize covariance for numerical stability.
                self.covs_[c] += 1e-6 * np.eye(d)

            # Convergence check.
            if it > 0 and abs(self.history_[-1] - self.history_[-2]) < self.tol:
                break

        self.n_iter_ = it + 1
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        n = X.shape[0]
        log_resp = np.zeros((n, self.k))
        for c in range(self.k):
            log_resp[:, c] = np.log(self.weights_[c] + 1e-300) + \
                              self._multivariate_normal_logpdf(X, self.means_[c], self.covs_[c])
        log_resp_max = log_resp.max(axis=1, keepdims=True)
        resp = np.exp(log_resp - log_resp_max)
        resp /= resp.sum(axis=1, keepdims=True)
        return resp

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


# Validate.
from sklearn.mixture import GaussianMixture

X_gmm, y_gmm = make_blobs(n_samples=400, centers=3, cluster_std=[1.0, 1.5, 0.5],
                           random_state=42, n_features=2)

mine = GMMScratch(k=3, random_state=0).fit(X_gmm)
theirs = GaussianMixture(n_components=3, covariance_type="full", random_state=0).fit(X_gmm)

# ARI to compare (labels arbitrary).
print(f"My ARI vs ground truth:       {adjusted_rand_score(y_gmm, mine.predict(X_gmm)):.3f}")
print(f"sklearn ARI vs ground truth:  {adjusted_rand_score(y_gmm, theirs.predict(X_gmm)):.3f}")
print(f"My n_iter:                    {mine.n_iter_}")


## 4.5 GMM vs K-Means on elliptical clusters

In [ ]:
# Generate clusters where GMM should clearly win — different shapes/orientations.
rng = np.random.default_rng(0)

def make_anisotropic_clusters():
    # 3 clusters with different shapes and orientations.
    X1 = rng.multivariate_normal([0, 0], [[1, 0.8], [0.8, 1]], size=150)
    X2 = rng.multivariate_normal([5, 5], [[2, -1], [-1, 1]], size=150)
    X3 = rng.multivariate_normal([0, 6], [[0.5, 0], [0, 3]], size=150)
    X = np.vstack([X1, X2, X3])
    y = np.array([0]*150 + [1]*150 + [2]*150)
    return X, y

X_aniso, y_aniso = make_anisotropic_clusters()

km = SKKMeans(n_clusters=3, n_init=10, random_state=0).fit(X_aniso)
gm = GaussianMixture(n_components=3, covariance_type="full", random_state=0).fit(X_aniso)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(X_aniso[:, 0], X_aniso[:, 1], c=y_aniso, cmap="tab10", s=20, edgecolors="black")
axes[0].set_title("Ground truth")
axes[1].scatter(X_aniso[:, 0], X_aniso[:, 1], c=km.labels_, cmap="tab10", s=20, edgecolors="black")
axes[1].set_title(f"K-Means (ARI={adjusted_rand_score(y_aniso, km.labels_):.3f})")
axes[2].scatter(X_aniso[:, 0], X_aniso[:, 1], c=gm.predict(X_aniso), cmap="tab10", s=20, edgecolors="black")
axes[2].set_title(f"GMM full cov (ARI={adjusted_rand_score(y_aniso, gm.predict(X_aniso)):.3f})")
plt.tight_layout()
plt.show()


**Read it:** when clusters are elongated and rotated, k-means draws straight cuts that ignore the shape. GMM with full covariance estimates each cluster's actual shape (ellipsoidal) and assigns points to whichever ellipsoid they're most likely under.

## 4.6 Cheat table

In [ ]:
gmm_cheat = pd.DataFrame([
    ["GMM (spherical)", "O(n · k · d · n_iter)",   "O(n · k · d)",   "Yes", "No", "Medium"],
    ["GMM (diagonal)",  "O(n · k · d · n_iter)",   "O(n · k · d)",   "Yes", "No", "Medium"],
    ["GMM (full)",      "O(n · k · d² · n_iter)",  "O(n · k · d²)",  "Yes", "No", "Medium"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
gmm_cheat


## 4.7 Interview Q&A

**Q: How is GMM different from k-means?**
> Two ways: (1) **Soft assignment** — GMM gives each point a probability of belonging to each cluster instead of a hard label. (2) **Cluster shape** — k-means assumes spherical clusters; GMM with full covariance models elliptical clusters. Conceptually, k-means is a special case of GMM (spherical covariance with equal weights and Mahalanobis → Euclidean).

**Q: Walk me through the EM algorithm.**
> E-step: given current parameters, compute **responsibilities** $\gamma_{ic}$ — the probability that point $i$ was generated by cluster $c$. M-step: re-estimate cluster parameters (means, covariances, weights) as the responsibility-weighted statistics. Iterate. Each iteration is guaranteed to **increase** the data log-likelihood, so convergence is monotonic. Like Lloyd's, you find a local maximum — multiple random restarts recommended.

**Q (trap): My GMM fit has a covariance matrix that's nearly singular. The log-likelihood blew up to infinity. What happened?**
> A cluster has collapsed onto a single data point or a degenerate line — its covariance is rank-deficient, and the Gaussian density at the mean → ∞. This is a **singularity** in the likelihood, and it's the main pathology of GMM. Fixes: (a) **regularize** by adding `epsilon * I` to each covariance matrix (sklearn's `reg_covar` parameter), (b) use a constrained covariance type (`spherical` or `diag`), or (c) **MAP** estimation with a Wishart prior.

**Q: How do you pick k for GMM?**
> Use information criteria: **BIC** (Bayesian Information Criterion) or **AIC** (Akaike Information Criterion). BIC penalizes more model parameters more heavily — prefer it for clustering. Compute BIC for k = 1, 2, 3, ... and pick the k that minimizes BIC. sklearn exposes `gmm.bic(X)`.

**Q (trap): K-Means gave me 3 clusters. GMM gave me 3 clusters. The cluster assignments are 95% the same. Did I gain anything from GMM?**
> Yes — the **probabilities**. K-means says "this point is in cluster 2." GMM says "this point is 0.85 in cluster 2, 0.12 in cluster 1, 0.03 in cluster 3." When the point is near a cluster boundary, that probability information is gold: it tells you which assignments are confident and which are borderline. For applications like A/B test routing, anomaly scoring, or downstream calibration, GMM's probabilities are far more useful than hard labels.

**Q: When does GMM fail?**
> Same failure modes as k-means plus singularities: (a) when clusters are non-Gaussian (curved manifolds → use DBSCAN/spectral), (b) when k is wrong, (c) when one cluster collapses to a single point (singularity), (d) on very high-dimensional data with full covariance (number of parameters is $O(k d^2)$ which is large).


---
# 5. Principal Component Analysis (PCA)

The #1 interview-tested dimensionality reduction algorithm. The from-scratch eigendecomposition implementation is canonical: if you can write PCA in 5 minutes, you've answered the dim-reduction question.

PCA finds a new orthogonal basis where the **first axis captures maximum variance**, the second axis captures the next-most variance (orthogonal to the first), and so on. Projecting onto the top $k$ axes gives the best $k$-dimensional linear approximation of the data.

## 5.1 Story

Given centered data $X \in \mathbb{R}^{n \times d}$ (subtract column means), find $\mathbf{w} \in \mathbb{R}^d$ that maximizes the **variance of the projection**:
$$\max_{\|\mathbf{w}\| = 1} \; \text{Var}(X \mathbf{w}) = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$$
where $\boldsymbol{\Sigma} = \frac{1}{n-1} X^\top X$ is the sample covariance matrix.

Lagrangian: $\mathcal{L} = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w} - \lambda (\mathbf{w}^\top \mathbf{w} - 1)$. Setting the gradient to zero:
$$\boldsymbol{\Sigma} \mathbf{w} = \lambda \mathbf{w}$$

**The principal components are the eigenvectors of the covariance matrix.** The variance along each axis is its corresponding eigenvalue.

## 5.2 Two equivalent routes

**Route 1 — eigendecomposition of the covariance matrix.**
```
1. Center X (subtract column means).
2. Compute covariance C = (1/(n-1)) * X^T X.
3. Eigendecompose: C = V Λ V^T  where V's columns are eigenvectors, Λ is diagonal of eigenvalues.
4. Sort eigenvalues descending; principal components = V's columns in that order.
```

**Route 2 — SVD of the data matrix (preferred numerically).**
```
1. Center X.
2. Compute SVD: X = U S V^T.
3. Principal components = columns of V; explained variance ∝ s_i².
```

SVD is more numerically stable and works even when $d > n$ (where $X^\top X$ is rank-deficient). sklearn uses SVD.

## 5.3 From-scratch implementation (both routes)

In [ ]:
class PCAScratch:
    '''
    PCA via eigendecomposition of the covariance matrix (or SVD of X).

    Time complexity:
      Eigendecomp route: O(d^3) for eigendecomp + O(n*d^2) for covariance
      SVD route:         O(min(n*d^2, n^2*d))
      transform:         O(n * d * k)
    '''
    def __init__(self, n_components=2, method="svd"):
        self.n_components = n_components
        self.method = method   # "svd" or "eig"

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        # 1. Center.
        self.mean_ = X.mean(axis=0)
        X_c = X - self.mean_

        if self.method == "eig":
            # 2. Covariance.
            cov = (X_c.T @ X_c) / (X.shape[0] - 1)
            # 3. Eigendecompose. eigh is for symmetric matrices — returns sorted ASCENDING.
            eigvals, eigvecs = np.linalg.eigh(cov)
            # Sort descending.
            order = np.argsort(eigvals)[::-1]
            eigvals = eigvals[order]
            eigvecs = eigvecs[:, order]
        else:
            # SVD route.
            # X_c = U @ diag(s) @ Vt
            U, s, Vt = np.linalg.svd(X_c, full_matrices=False)
            # Explained variance = s^2 / (n-1)
            eigvals = (s ** 2) / (X.shape[0] - 1)
            eigvecs = Vt.T

        # Keep top n_components.
        self.components_ = eigvecs[:, :self.n_components].T   # shape: (k, d)
        self.explained_variance_ = eigvals[:self.n_components]
        total_var = eigvals.sum()
        self.explained_variance_ratio_ = self.explained_variance_ / total_var
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) @ self.components_.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_


# Generate correlated 3D data.
rng = np.random.default_rng(0)
X_pca_demo = rng.multivariate_normal(
    mean=[0, 0, 0],
    cov=[[3, 2, 1], [2, 2, 0.5], [1, 0.5, 1]],
    size=300,
)

# Both routes should give the same result.
pca_eig = PCAScratch(n_components=2, method="eig").fit(X_pca_demo)
pca_svd = PCAScratch(n_components=2, method="svd").fit(X_pca_demo)

# Components can differ by sign — that's fine, the subspace is the same.
print("Eigendecomposition route:")
print(f"  Explained variance: {pca_eig.explained_variance_.round(3)}")
print(f"  Variance ratio:     {pca_eig.explained_variance_ratio_.round(3)}")

print("\nSVD route:")
print(f"  Explained variance: {pca_svd.explained_variance_.round(3)}")
print(f"  Variance ratio:     {pca_svd.explained_variance_ratio_.round(3)}")

# Validate against sklearn.
from sklearn.decomposition import PCA
sk_pca = PCA(n_components=2).fit(X_pca_demo)
print("\nsklearn:")
print(f"  Explained variance: {sk_pca.explained_variance_.round(3)}")
print(f"  Variance ratio:     {sk_pca.explained_variance_ratio_.round(3)}")


## 5.4 Visualization — projecting MNIST-like data

In [ ]:
# Use a digits-like dataset to visualize PCA reducing high-dimensional structure.
from sklearn.datasets import load_digits

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

# Standardize before PCA (always do this if features have different scales).
from sklearn.preprocessing import StandardScaler
X_digits_s = StandardScaler().fit_transform(X_digits)

# Project to 2D.
pca_2d = PCA(n_components=2).fit(X_digits_s)
Z = pca_2d.transform(X_digits_s)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap="tab10", s=15, edgecolors="black", linewidth=0.3)
plt.colorbar(scatter, ax=ax, label="Digit")
ax.set_xlabel(f"PC 1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC 2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("MNIST digits — PCA to 2D (linear projection)")
plt.show()


**Read it:** 2D PCA captures ~22% of the variance — that's why digits don't fully separate. Some digits (0, 6) cluster cleanly; others (3, 5, 8) overlap because they share strokes. **PCA is a LINEAR projection** — it can't separate classes that need non-linear boundaries. For visualization of complex structure, t-SNE/UMAP usually do better.

## 5.5 The explained variance curve — picking the number of components

In [ ]:
# Fit PCA with ALL components and plot cumulative explained variance.
pca_full = PCA().fit(X_digits_s)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cum_var, marker=".", markersize=3)
ax.axhline(0.90, color="red", linestyle="--", label="90% variance")
ax.axhline(0.95, color="orange", linestyle="--", label="95% variance")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance ratio")
ax.set_title("How many PCA components to keep?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

# How many components to hit 95%?
k_95 = np.argmax(cum_var >= 0.95) + 1
print(f"Components needed for 90% variance: {np.argmax(cum_var >= 0.90) + 1} of {X_digits_s.shape[1]}")
print(f"Components needed for 95% variance: {k_95} of {X_digits_s.shape[1]}")


**The decision:** typical thresholds are 90% (for visualization / aggressive compression) or 95% (for downstream modeling). Pick based on the downstream task — if predictive accuracy matters, do a CV with different `n_components` rather than picking by variance alone.

## 5.6 When PCA hurts

PCA assumes:
1. **Linearity** — captures linear directions of variance. Curved structure (Swiss roll, MNIST classes) is lost.
2. **Variance ≈ information** — the largest-variance direction is most "important." Often false. Example: in a fraud-detection dataset, the fraud signal might be a low-variance feature (rare event); PCA discards it.
3. **Orthogonal components** — interpretability suffers when the true factors are correlated.

**When NOT to use PCA:**
- **Classification tasks where rare classes matter** — variance ≠ predictiveness.
- **Non-linear manifold structure** — use t-SNE, UMAP, kernel PCA, or autoencoders.
- **Interpretability matters** — PCA components are linear combinations of all features, hard to name.

## 5.7 Cheat table

In [ ]:
pca_cheat = pd.DataFrame([
    ["PCA (eigendecomp)", "O(n·d² + d³)", "O(n·d·k)",
     "Yes", "No", "Medium (linear combo of features)"],
    ["PCA (SVD)",         "O(min(n·d², n²·d))", "O(n·d·k)",
     "Yes", "No", "Same"],
    ["Truncated SVD",     "O(n·d·k)", "O(n·d·k)",
     "Yes", "Yes (works on sparse)", "Same"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
pca_cheat


## 5.8 Interview Q&A

**Q: Walk me through PCA mathematically.**
> Given centered data $X$, find unit vector $\mathbf{w}$ that maximizes the variance $\mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$. Lagrangian gives $\boldsymbol{\Sigma} \mathbf{w} = \lambda \mathbf{w}$ — so principal components are **eigenvectors of the covariance matrix**. Sort eigenvalues descending; the top $k$ eigenvectors are the top $k$ principal components, and their eigenvalues are the variance captured along each.

**Q: Why use SVD instead of eigendecomposition?**
> Three reasons: (1) **Numerical stability** — SVD doesn't form $X^\top X$ (which squares condition number). (2) Works when $d > n$ — covariance is singular but SVD still works. (3) **Efficiency** — for $n \gg d$, computing covariance + eigendecomp is $O(nd^2 + d^3)$; SVD on $X$ is the same complexity but with better constants. sklearn uses SVD by default.

**Q (trap): Should I scale features before PCA?**
> Almost always yes. PCA maximizes variance; if one feature has variance 1 and another variance 100, the second dominates the first principal component **regardless of how informative it is**. Standardize features (zero mean, unit variance) before PCA. The exception is when features are already on the same scale (e.g., pixel intensities all in [0, 255]) — then scaling isn't necessary but doesn't hurt.

**Q: How do you pick the number of components?**
> Two approaches: (a) **Variance threshold** — keep enough components to retain 90% or 95% of variance. Simple, no downstream task needed. (b) **CV on downstream task** — if PCA is preprocessing for a classifier, pick `n_components` via cross-validation on classifier accuracy. The second is better when accuracy matters; the first is faster for exploratory analysis.

**Q (trap): I ran PCA on my fraud-detection dataset and accuracy got worse. Why?**
> Variance ≠ predictiveness. The fraud signal might live in a **low-variance** direction — rare events have small marginal variance but high predictive value. PCA discards low-variance directions, including this one. For classification, prefer **supervised** dimensionality reduction (LDA, or just train the classifier on raw features). PCA is best when variance and information are correlated, which is true for images and many physics problems but often false for tabular data.

**Q: What's the difference between PCA and Linear Discriminant Analysis (LDA)?**
> **PCA is unsupervised** — finds directions of maximum variance, ignoring labels. **LDA is supervised** — finds directions of maximum **class separation** (maximize between-class variance, minimize within-class variance). For classification, LDA is usually strictly better as a preprocessing step. PCA wins when you don't have labels.

**Q (trap): I PCA-reduced my data to 2 dimensions but the classes still overlap. Should I conclude my classes aren't separable?**
> No. PCA is **linear**. If your classes are separable by a non-linear boundary (which is common), the 2D linear projection can't show that separation. Run t-SNE or UMAP instead — those are non-linear and will reveal separability if it exists. PCA's failure to separate classes is a property of the projection, not the data.


---
# 6. t-SNE (t-distributed Stochastic Neighbor Embedding)

The go-to **visualization** tool for high-dimensional data. Preserves **local neighborhoods** (points close in high-D stay close in 2D) while letting global distances distort. Famously revealing for visualizing clusters in MNIST, single-cell RNA-seq, word embeddings.

**Not a clustering algorithm. Not a feature for downstream modeling. Purely visualization.**

## 6.1 The intuition

For each pair of high-dimensional points $(\mathbf{x}_i, \mathbf{x}_j)$, define $p_{ij}$ = probability that $i$ would pick $j$ as its neighbor under a Gaussian centered at $i$. For each pair of low-dimensional points $(\mathbf{y}_i, \mathbf{y}_j)$, define $q_{ij}$ = same under a **t-distribution** with one degree of freedom (heavy tails).

Then minimize the **KL divergence** between $P$ and $Q$:
$$\text{KL}(P \| Q) = \sum_{i \ne j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

Gradient descent in $Y$ space. The asymmetry of KL makes t-SNE preserve neighborhoods: it heavily penalizes putting close high-D pairs far in low-D, but only mildly penalizes putting far high-D pairs close in low-D.

## 6.2 Why the t-distribution in low-D?

The t-distribution with 1 dof (Cauchy distribution) has **heavy tails** compared to a Gaussian. This solves the "crowding problem": in 2D you can't fit as many neighbors at a given distance as you can in high-D. The heavy tails let distant pairs sit farther apart, freeing space to keep close pairs close.

## 6.3 Perplexity — the one knob

**Perplexity** is roughly the effective number of neighbors each point considers in the high-D similarity computation. Set per-point Gaussian bandwidth $\sigma_i$ so that the Shannon entropy of $p_{j|i}$ equals $\log_2(\text{perplexity})$.

- **Low perplexity (5-15):** focuses on very local structure; preserves micro-clusters.
- **High perplexity (50-100):** preserves global structure better; smoother.
- **Default:** 30. Try 5, 30, 100 on the same data and see which best reveals what you care about.

## 6.4 Conceptual mechanics — DO NOT implement from scratch in an interview

The actual algorithm involves:
- Perplexity calibration via binary search on $\sigma_i$ for each point.
- Symmetrization of the joint probabilities.
- Gradient descent with momentum on a non-convex loss.
- Barnes-Hut $O(n \log n)$ approximation for large $n$.

This is roughly 500 lines of careful code. **Don't try to implement it from scratch.** Understand the math + use sklearn. Asked in interviews as a **comparison question** (PCA vs t-SNE), not as a "write it" question.

## 6.5 t-SNE on MNIST digits

In [ ]:
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits

digits = load_digits()
X_digits, y_digits = digits.data, digits.target

# Standardize, then PCA to ~30 dims as a smart preprocessor.
# This is the standard recipe — t-SNE on raw high-dim data is slow and noisy.
from sklearn.preprocessing import StandardScaler
X_s = StandardScaler().fit_transform(X_digits)
pca_pre = PCA(n_components=30).fit_transform(X_s)

# t-SNE.
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=0,
            init="pca", learning_rate="auto")
Z_tsne = tsne.fit_transform(pca_pre)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PCA 2D for comparison.
pca_2d = PCA(n_components=2).fit_transform(X_s)
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], c=y_digits, cmap="tab10", s=15, edgecolors="black", linewidth=0.3)
axes[0].set_title("PCA (linear) — classes overlap")
axes[0].set_xlabel("PC 1"); axes[0].set_ylabel("PC 2")

axes[1].scatter(Z_tsne[:, 0], Z_tsne[:, 1], c=y_digits, cmap="tab10", s=15, edgecolors="black", linewidth=0.3)
axes[1].set_title(f"t-SNE — classes separate (perplexity=30)")
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")

plt.tight_layout()
plt.show()


**Read it side by side:** PCA squeezes the 10 digit classes into overlapping blobs because it's a linear projection. t-SNE reveals 10 distinct islands because it preserves local neighborhood structure non-linearly. This is the canonical "why t-SNE for visualization" picture.

## 6.6 The perplexity grid — t-SNE's main failure mode

In [ ]:
# t-SNE is sensitive to perplexity. Show the same data at 3 perplexity values.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, perp in zip(axes, [5, 30, 100]):
    tsne = TSNE(n_components=2, perplexity=perp, max_iter=500, random_state=0,
                init="pca", learning_rate="auto")
    Z = tsne.fit_transform(pca_pre)
    ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap="tab10", s=12, edgecolors="black", linewidth=0.2)
    ax.set_title(f"perplexity = {perp}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


**Read it:** perplexity 5 fragments classes into sub-clusters. Perplexity 100 starts blending classes. Perplexity 30 is the sweet spot here. **Always try multiple perplexity values** — t-SNE is famously seed-and-perplexity sensitive.

## 6.7 Cheat table

In [ ]:
tsne_cheat = pd.DataFrame([
    ["t-SNE (exact)",      "O(n²)",     "No predict method (can't embed new points)",
     "Yes", "No", "Low (axes have no meaning)"],
    ["t-SNE (Barnes-Hut)", "O(n log n)", "Same",
     "Yes", "No", "Same"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
tsne_cheat


## 6.8 Interview Q&A — t-SNE

**Q: What's the difference between PCA and t-SNE?**
> **PCA:** linear, global, fast, deterministic, has inverse_transform, captures variance. Good for preprocessing for downstream models, mild compression, decorrelation. **t-SNE:** non-linear, local, slow, stochastic, no inverse_transform, captures neighborhood structure. Good ONLY for visualization. Don't use t-SNE features for downstream modeling — the embedding is unstable across runs and doesn't generalize to new points.

**Q (trap): Can I use t-SNE features as input to a classifier?**
> No, for several reasons: (1) t-SNE has no `transform` method for new data — the embedding is computed during fit, doesn't generalize. (2) The embedding is **stochastic** — different runs give different 2D positions, so the "features" aren't stable. (3) t-SNE preserves local structure but distorts global distances arbitrarily — it has no consistent geometric meaning. For modeling, use UMAP (which has `transform`) or train end-to-end. t-SNE is **visualization only**.

**Q: What does t-SNE's "perplexity" parameter control?**
> Roughly the **effective number of neighbors** each point considers when computing high-D similarities. Low perplexity (5-10) emphasizes very local structure. High perplexity (50-100) emphasizes more global structure. Default 30 works well most of the time. **t-SNE is sensitive to perplexity** — always try multiple values.

**Q (trap): In my t-SNE plot, two clusters are 10 units apart. Does that mean they're "twice as different" as two clusters 5 units apart?**
> No. **t-SNE distances are meaningless on a global scale.** Only local neighborhood relationships are preserved. The size of gaps between clusters in t-SNE is essentially arbitrary; cluster compactness is also distorted. Don't draw conclusions like "class A is more similar to class B than class C" from a t-SNE plot — only "points within class A are close together."

**Q: Why preprocess with PCA before t-SNE?**
> Two reasons: (1) **Speed** — t-SNE is O(n²) in exact mode. Reducing dimensionality first (e.g., to 30-50 PCA components) is much faster than running t-SNE on raw 784-dim images. (2) **Denoising** — PCA discards low-variance directions which are often noise. Cleaner input → cleaner t-SNE visualization. Standard pipeline: standardize → PCA to 30-50 dims → t-SNE to 2 dims.

**Q: Can t-SNE recover the number of clusters in the data?**
> Sometimes, but you can't trust it. If t-SNE produces 7 visible blobs, you might have 7 clusters — or 5 clusters that t-SNE split, or 10 clusters that t-SNE merged. The number of visible groups depends on perplexity and the random seed. Use a real clustering algorithm (k-means, GMM, DBSCAN) on the original or PCA-reduced data; use t-SNE only to **visualize** what the cluster algorithm found.


---
# 7. UMAP (Uniform Manifold Approximation and Projection)

The modern alternative to t-SNE. Faster, scales to millions of points, has a `transform` method for new data, generally preserves **more global structure** than t-SNE while keeping local structure. The default for single-cell biology and large-scale embedding visualization in 2026.

UMAP came out in 2018 (McInnes et al.); since 2020 it has steadily replaced t-SNE in most pipelines. If you reach for t-SNE today, ask yourself first whether UMAP would be better.

## 6.1 The intuition (high-level)

UMAP is grounded in topological theory (simplicial sets, fuzzy graphs), but the operational idea is the same flavor as t-SNE:

1. Build a **weighted neighborhood graph** of the high-dimensional data. Each point's edges go to its $k$ nearest neighbors, weighted by a fuzzy-set membership function that adapts to local density.
2. Optimize a low-dimensional embedding so that the **fuzzy graph in low-D matches the high-D graph** — using cross-entropy as the loss (different from t-SNE's KL).
3. Use **stochastic gradient descent** with negative sampling — much faster than t-SNE's full gradient.

The result: similar local structure to t-SNE, better global structure (distances between clusters are more meaningful), much faster (handles 1M+ points where t-SNE caps out around 100K).

## 7.2 Key parameters

- `n_neighbors` (default 15) — analogous to perplexity. Smaller = more local. 5-50 typical.
- `min_dist` (default 0.1) — minimum distance between points in low-D. Larger → more spread-out clusters. Smaller → tighter clusters.
- `metric` (default `euclidean`) — works with cosine, Manhattan, custom metrics.

## 7.3 UMAP vs t-SNE on the same data

In [ ]:
import umap

# Use the same PCA-preprocessed digits data from earlier.
um = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=0)
Z_umap = um.fit_transform(pca_pre)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(Z_tsne[:, 0], Z_tsne[:, 1], c=y_digits, cmap="tab10", s=15, edgecolors="black", linewidth=0.3)
axes[0].set_title("t-SNE")
axes[0].set_xticks([]); axes[0].set_yticks([])

axes[1].scatter(Z_umap[:, 0], Z_umap[:, 1], c=y_digits, cmap="tab10", s=15, edgecolors="black", linewidth=0.3)
axes[1].set_title("UMAP")
axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout()
plt.show()


**Read the comparison:** both reveal the 10 digit classes well. UMAP usually shows **more separation between distinct classes** and **tighter intra-class structure**. Distances between cluster centers in UMAP are more meaningful than in t-SNE — when class A and B are far apart in UMAP, they're genuinely far in the original space.

## 7.4 The killer feature — UMAP can embed new points

In [ ]:
# UMAP has a transform() method that t-SNE lacks.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(pca_pre, y_digits, test_size=0.3, random_state=0)

# Fit UMAP on training data only.
um = umap.UMAP(n_components=2, random_state=0).fit(X_train)

# Embed both train and test using the SAME fitted model.
Z_train = um.embedding_           # already computed during fit
Z_test  = um.transform(X_test)    # NEW points embedded by the fitted model

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(Z_train[:, 0], Z_train[:, 1], c=y_train, cmap="tab10",
            s=12, alpha=0.4, label="Train")
ax.scatter(Z_test[:, 0],  Z_test[:, 1],  c=y_test,  cmap="tab10",
            s=40, marker="x", edgecolors="black", label="Test (new points)")
ax.set_title("UMAP can embed new points via transform()")
ax.legend()
plt.show()


**This is the killer feature:** UMAP can be used as a **preprocessing step** in a real ML pipeline — fit on training data, transform train + val + test, downstream model uses the low-dim features. t-SNE can't do this; its embedding is tied to the specific dataset it was fit on.

## 7.5 Cheat table — t-SNE vs UMAP

In [ ]:
tsne_umap_compare = pd.DataFrame([
    ["Speed",                "Slow — O(n²) exact, O(n log n) Barnes-Hut", "Fast — handles 1M+ points"],
    ["Embed new points?",    "No",                                          "Yes (transform method)"],
    ["Preserves local",      "Yes (best in class)",                         "Yes"],
    ["Preserves global",     "Poor — distances meaningless globally",       "Good — distances more meaningful"],
    ["Stochastic?",          "Yes",                                         "Yes, but more stable"],
    ["Stability across seeds", "Low",                                       "Higher"],
    ["Use for downstream features?", "No",                                  "Yes"],
    ["Default in 2026?",     "For specialized cases (legacy)",              "Yes — preferred default"],
], columns=["Property", "t-SNE", "UMAP"])
tsne_umap_compare


## 7.6 Interview Q&A — UMAP

**Q: When would you choose UMAP over t-SNE?**
> Almost always in 2026 unless you have a specific reason for t-SNE. UMAP is faster, scales to millions of points, has a `transform()` method (so you can embed new data), preserves global structure better, and is more stable across runs. t-SNE is still slightly better at revealing fine local sub-structure within clusters; that's the main remaining edge.

**Q (trap): I ran UMAP twice with the same data and got slightly different embeddings. Is something wrong?**
> No, UMAP is **stochastic** (uses SGD with negative sampling). Different runs give slightly different embeddings unless you fix `random_state`. The overall structure should be very similar; only fine details vary. If you need reproducibility, always pass `random_state`. t-SNE is even more sensitive to seeds.

**Q: What's the analog of t-SNE's perplexity in UMAP?**
> `n_neighbors`. Small `n_neighbors` (5-10) emphasizes local structure — many small islands. Large `n_neighbors` (50-100) emphasizes global structure — fewer, broader clusters. Default 15. Like perplexity, it's the main knob for tuning UMAP visualization.

**Q (trap): Can I use UMAP for clustering directly, like reading clusters off the 2D embedding?**
> No — that's the same anti-pattern as with t-SNE. UMAP visualizes structure; running k-means or DBSCAN on the 2D embedding can artifact-cluster. The right pattern: (a) cluster in the ORIGINAL (or PCA-reduced ~30-dim) space, then (b) visualize the clusters with UMAP. Don't conflate "what UMAP shows" with "what's actually there."

**Q: What's UMAP's main advantage over t-SNE in production ML pipelines?**
> The `transform()` method. UMAP can be **fit on training data and applied to new data** with the same learned mapping. This makes UMAP a viable preprocessing step in real pipelines (fit on train, transform train+val+test, downstream model uses low-dim features). t-SNE can only be re-fit from scratch on the new data, giving inconsistent embeddings.


---
# 8. TF-IDF (Term Frequency × Inverse Document Frequency)

The classic text representation. Convert documents to numerical vectors by weighting each word by **how often it appears in this document** times **how rare it is across the corpus**. The result: common words ("the," "and") get low weight; words that distinguish a document get high weight.

Despite being from 1972, TF-IDF + linear classifier is still **the right baseline** for any text classification problem. Sentence embeddings beat it on hard tasks but cost 100-1000x more compute. Always implement TF-IDF first.

## 8.1 Core math

**Term frequency** of word $t$ in document $d$:
$$\text{tf}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

(Or just raw count; or log-scaled $1 + \log(\text{count})$.)

**Inverse document frequency** of word $t$ across corpus of $N$ documents:
$$\text{idf}(t) = \log \frac{N}{\text{# docs containing } t}$$

(Or smoothed: $\log \frac{1 + N}{1 + \text{df}(t)} + 1$ — sklearn's default to avoid zero division.)

**TF-IDF score:**
$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t)$$

Each document becomes a sparse vector over the vocabulary. Documents are usually L2-normalized so they live on the unit sphere, making cosine similarity equivalent to dot product.

## 8.2 From-scratch implementation

In [ ]:
class TfidfVectorizerScratch:
    '''
    From-scratch TF-IDF vectorizer. Matches sklearn's TfidfVectorizer
    (with smooth_idf=True, sublinear_tf=False, norm='l2').

    Time complexity:
      fit:       O(total_tokens) for building vocab and DF counts
      transform: O(total_tokens)
    '''
    def __init__(self, lowercase=True):
        self.lowercase = lowercase

    @staticmethod
    def _tokenize(text):
        '''Same regex sklearn uses by default: words of 2+ chars.'''
        import re
        return re.findall(r"(?u)\b\w\w+\b", text)

    def fit(self, docs):
        if self.lowercase:
            docs = [d.lower() for d in docs]
        # Build vocabulary and document frequency.
        df_counter = {}
        for doc in docs:
            tokens = set(self._tokenize(doc))   # set: each token counted once per doc
            for tok in tokens:
                df_counter[tok] = df_counter.get(tok, 0) + 1

        self.vocabulary_ = {tok: i for i, tok in enumerate(sorted(df_counter))}
        self.n_features_ = len(self.vocabulary_)
        N = len(docs)

        # Smoothed IDF (sklearn default).
        # idf(t) = log((1 + N) / (1 + df(t))) + 1
        self.idf_ = np.zeros(self.n_features_)
        for tok, idx in self.vocabulary_.items():
            self.idf_[idx] = np.log((1 + N) / (1 + df_counter[tok])) + 1.0
        return self

    def transform(self, docs):
        if self.lowercase:
            docs = [d.lower() for d in docs]
        n = len(docs)
        # Build sparse term-count matrix, then multiply by IDF, then L2 normalize.
        # For small corpora, dense is fine.
        X = np.zeros((n, self.n_features_))
        for i, doc in enumerate(docs):
            for tok in self._tokenize(doc):
                if tok in self.vocabulary_:
                    X[i, self.vocabulary_[tok]] += 1
        # TF (raw counts, since sublinear_tf=False) × IDF
        X = X * self.idf_
        # L2 normalize rows.
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return X / norms

    def fit_transform(self, docs):
        return self.fit(docs).transform(docs)


# Validate against sklearn.
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "the quick brown fox jumps over the lazy dog",
    "never gonna give you up never gonna let you down",
    "the dog ate my homework",
    "homework is the bane of my existence",
    "give me a quick fox or give me death",
]

mine = TfidfVectorizerScratch()
X_mine = mine.fit_transform(corpus)

theirs = TfidfVectorizer()
X_theirs = theirs.fit_transform(corpus).toarray()

print(f"My shape: {X_mine.shape}, sklearn shape: {X_theirs.shape}")
print(f"My vocab size: {len(mine.vocabulary_)}, sklearn vocab size: {len(theirs.vocabulary_)}")

# Vocabularies might differ in ordering. Compare per-token IDF values.
common_tokens = sorted(set(mine.vocabulary_) & set(theirs.vocabulary_))
my_idf = [mine.idf_[mine.vocabulary_[t]] for t in common_tokens]
their_idf = [theirs.idf_[theirs.vocabulary_[t]] for t in common_tokens]
print(f"\nIDF values match exactly? {np.allclose(my_idf, their_idf)}")


## 8.3 TF-IDF for classification — the canonical baseline

In [ ]:
# Build a synthetic spam classifier with TF-IDF + Logistic Regression.
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

docs = [
    # Spam
    "free money click here win prize today",
    "claim your free gift cash prize instantly",
    "win lottery cash money no risk",
    "your prize is waiting click the link",
    "free instant cash for you sign up now",
    "winner winner chicken dinner free prize",
    "buy buy buy discount price expires",
    "limited time offer act now huge savings",
    # Ham
    "meeting tomorrow at 10am in conference room",
    "please review the attached report by friday",
    "lunch with the team on monday",
    "project deadline is next week reminder",
    "the documentation has been updated please check",
    "happy birthday hope you have a great day",
    "let me know when you are free for a call",
    "could you send me the updated spreadsheet",
] * 30
labels = ([1]*8 + [0]*8) * 30

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=2)),
    ("clf",   LogisticRegression(max_iter=1000)),
])
pipe.fit(docs, labels)

print(f"Training accuracy: {pipe.score(docs, labels):.3f}")

# Inspect the top spam-indicating features (largest positive coefficients).
vocab = pipe.named_steps["tfidf"].get_feature_names_out()
coef = pipe.named_steps["clf"].coef_[0]
top_spam_idx = np.argsort(coef)[-8:][::-1]
top_ham_idx  = np.argsort(coef)[:8]

print("\nTop SPAM-indicating words:")
for i in top_spam_idx:
    print(f"  {vocab[i]:<14s} weight = {coef[i]:+.2f}")

print("\nTop HAM-indicating words:")
for i in top_ham_idx:
    print(f"  {vocab[i]:<14s} weight = {coef[i]:+.2f}")


## 8.4 Important TF-IDF knobs

| Parameter | What it does |
|---|---|
| `min_df` | Minimum doc frequency. Drop words appearing in too few docs (noise). `min_df=2` is common. |
| `max_df` | Maximum doc frequency. Drop words appearing in too many docs (stopwords-like). `max_df=0.95` drops words in 95%+ of docs. |
| `ngram_range` | `(1, 2)` adds bigrams. Captures "not good" vs "good." Triples vocabulary size. |
| `sublinear_tf` | Replace TF with `1 + log(TF)` — diminishes effect of repeated words. Often helps. |
| `stop_words` | `"english"` removes common stopwords. Mixed results — sometimes useful, sometimes drops signal. |
| `analyzer='char_wb'` with `ngram_range=(3,5)` | Character n-grams. Robust to typos. Useful for non-English or noisy text. |

**Default starting recipe:** `TfidfVectorizer(min_df=2, ngram_range=(1, 2), sublinear_tf=True)`.

## 8.5 Cheat table

In [ ]:
tfidf_cheat = pd.DataFrame([
    ["TF-IDF", "O(total_tokens)", "O(total_tokens)",
     "No (output is normalized)", "No", "High (per-feature weights interpretable)"],
], columns=["Algorithm", "Train time", "Predict time", "Needs scaling?", "Handles missing?", "Interpretability"])
tfidf_cheat


## 8.6 Interview Q&A

**Q: Why does TF-IDF work? What's the intuition?**
> Two-part scoring: TF gives weight to words **used a lot in this document** (signal it's about that topic). IDF down-weights words **common across many documents** (which carry little discriminative info — "the," "and"). Together: a word is important if it's frequent in this document AND rare overall. It's a simple model but captures the basic property of "informative words."

**Q: When is TF-IDF still relevant in 2026, when we have sentence embeddings?**
> Three cases: (1) **Baselines** — always run TF-IDF + linear classifier before reaching for embeddings. Often within a few % of fancy models on classification tasks, at 1/1000 the compute. (2) **Interpretability** — TF-IDF + linear model has per-word weights you can read directly. (3) **Resource constraints** — running BERT for billions of documents is expensive; TF-IDF is essentially free.

**Q (trap): Should I scale TF-IDF features before feeding to logistic regression?**
> No — sklearn's TfidfVectorizer **already L2-normalizes** each document vector. Each row has unit norm. Adding StandardScaler on top can hurt by destroying the sparsity structure (StandardScaler will subtract mean, making everything dense). Just use TF-IDF output directly.

**Q: TF-IDF vs Bag of Words (raw counts) — when does TF-IDF help?**
> TF-IDF helps when the corpus has very common uninformative words (which is most corpora). Bag of Words gives "the" the same weight as "thermonuclear"; TF-IDF correctly down-weights "the." In practice TF-IDF almost always equals or beats raw counts; default to TF-IDF. The exception is when you're computing exact term statistics for downstream use (e.g., language modeling).

**Q (trap): My TF-IDF + logistic regression gets 95% accuracy on training, 65% on test. What's wrong?**
> Vocabulary mismatch is likely: words in training that aren't in test (and vice versa) — TF-IDF has fixed vocabulary, so unknown words are dropped. Try (a) `min_df=2` to drop ultra-rare words from training (which don't generalize anyway), (b) `max_features=10000` to cap vocabulary size, (c) check for **train/test distribution shift** — is your training data systematically different from test? Also: 95→65 gap suggests heavy regularization is missing. Increase `LogisticRegression(C=0.1)` or smaller.

**Q: What's the difference between TF-IDF and word embeddings (word2vec)?**
> TF-IDF: per-document **sparse** vector, dimensions = vocabulary size. Each dimension corresponds to one word, value = TF-IDF weight. word2vec: per-WORD **dense** vector, dimensions = ~300 (chosen by you). Captures semantic similarity (king - man + woman ≈ queen). To represent a **document**, average the word vectors (or use a more sophisticated pooling). word2vec sees synonyms; TF-IDF doesn't (it sees "car" and "automobile" as different features).


---
# 9. Collaborative Filtering (Recommender Systems)

The "users who liked this also liked..." algorithm. Given a sparse user-item rating matrix, predict missing ratings. Three flavors:

1. **User-user CF** — recommend items liked by users similar to you.
2. **Item-item CF** — recommend items similar to ones you've liked.
3. **Matrix factorization** — decompose the rating matrix into low-rank user and item factor matrices; predict via dot product.

Item-item and matrix factorization are the production-grade approaches. User-user is the conceptual entry point.

## 9.1 Setup — synthetic rating data

A small movie ratings dataset (10 users × 12 movies) with realistic missing-value pattern.

In [ ]:
# Build a synthetic user-item rating matrix.
# 10 users, 12 movies. Sparse: each user rates 4-7 movies.
np.random.seed(42)
n_users, n_items = 10, 12

# True latent factors (2 genres: action vs romance).
user_factors_true = np.random.randn(n_users, 2)
item_factors_true = np.random.randn(n_items, 2)
true_ratings = user_factors_true @ item_factors_true.T   # (10, 12)

# Rescale to a 1-5 rating scale.
true_ratings = 3 + 1.5 * true_ratings
true_ratings = np.clip(true_ratings, 1, 5)

# Make some ratings missing (sparse observations).
R = np.full((n_users, n_items), np.nan)
for u in range(n_users):
    rated_idx = np.random.choice(n_items, size=np.random.randint(4, 8), replace=False)
    R[u, rated_idx] = true_ratings[u, rated_idx] + 0.3 * np.random.randn(len(rated_idx))
    R[u, rated_idx] = np.clip(R[u, rated_idx], 1, 5).round(1)

users = [f"User{i}" for i in range(n_users)]
items = [f"Movie{j}" for j in range(n_items)]
R_df = pd.DataFrame(R, index=users, columns=items)
print("Rating matrix (NaN = unrated):")
R_df.round(1)


## 9.2 User-user collaborative filtering

In [ ]:
def user_user_cf(R, user_idx, item_idx, k=3):
    '''
    Predict rating of user `user_idx` for item `item_idx` using user-user CF.

    Steps:
    1. Compute cosine similarity between target user and all other users
       (using only items they've BOTH rated).
    2. Take the top-k most similar users who HAVE rated item_idx.
    3. Predicted rating = weighted average of their ratings, weighted by similarity.
    '''
    target_ratings = R[user_idx]
    n_users = R.shape[0]

    # Compute similarity to all other users.
    sims = np.zeros(n_users)
    for u in range(n_users):
        if u == user_idx:
            continue
        # Use only items both users have rated.
        mask = ~np.isnan(target_ratings) & ~np.isnan(R[u])
        if mask.sum() < 2:
            continue
        a, b = target_ratings[mask], R[u][mask]
        # Mean-center each user (each user has their own rating tendency).
        a = a - a.mean()
        b = b - b.mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        sims[u] = (a @ b) / denom if denom > 0 else 0

    # Pick top-k similar users who have rated this item.
    candidates = [(u, sims[u]) for u in range(n_users)
                  if u != user_idx and not np.isnan(R[u, item_idx])]
    candidates.sort(key=lambda x: -x[1])
    top_k = candidates[:k]

    if not top_k:
        return np.nanmean(R[user_idx])   # fallback: user's mean

    # Weighted prediction (with mean-centering, since users have different rating baselines).
    user_mean = np.nanmean(R[user_idx])
    weighted_diff_sum, weight_sum = 0.0, 0.0
    for u, s in top_k:
        if s <= 0:
            continue
        u_mean = np.nanmean(R[u])
        weighted_diff_sum += s * (R[u, item_idx] - u_mean)
        weight_sum += s
    if weight_sum == 0:
        return user_mean
    return user_mean + weighted_diff_sum / weight_sum


# Predict User0's rating for a movie they didn't rate.
user, item = 0, 11
true = true_ratings[user, item]
predicted = user_user_cf(R, user, item, k=3)

print(f"True rating of {users[user]} for {items[item]}:        {true:.2f}")
print(f"Predicted via user-user CF (k=3 neighbors):  {predicted:.2f}")
print(f"Actual recorded rating (may be NaN):         {R[user, item]}")


## 9.3 Item-item collaborative filtering

**Item-item is the production default** for two reasons: (1) item-item similarities are more stable than user-user (items don't change behavior over time; users do). (2) items are usually fewer than users (Amazon: hundreds of millions of users, tens of millions of items), so item-item similarity matrices are smaller and cheaper.

In [ ]:
def item_item_cf(R, user_idx, item_idx, k=3):
    '''
    Predict rating using item-item CF.

    Steps:
    1. Find items similar to item_idx (cosine similarity over users who rated both).
    2. Among items the user HAS rated, take the top-k most similar to item_idx.
    3. Predicted rating = weighted average of user's ratings for those, weighted by similarity.
    '''
    n_items = R.shape[1]

    # Similarity of target item to all other items.
    sims = np.zeros(n_items)
    for j in range(n_items):
        if j == item_idx:
            continue
        mask = ~np.isnan(R[:, item_idx]) & ~np.isnan(R[:, j])
        if mask.sum() < 2:
            continue
        a, b = R[mask, item_idx], R[mask, j]
        a = a - a.mean()
        b = b - b.mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        sims[j] = (a @ b) / denom if denom > 0 else 0

    # Items the user has rated.
    rated_items = [j for j in range(n_items) if j != item_idx and not np.isnan(R[user_idx, j])]
    rated_items.sort(key=lambda j: -sims[j])
    top_k = rated_items[:k]

    if not top_k or all(sims[j] <= 0 for j in top_k):
        return np.nanmean(R[user_idx])

    num = sum(sims[j] * R[user_idx, j] for j in top_k if sims[j] > 0)
    den = sum(sims[j] for j in top_k if sims[j] > 0)
    return num / den if den > 0 else np.nanmean(R[user_idx])


user, item = 0, 11
print(f"True rating:                                 {true_ratings[user, item]:.2f}")
print(f"User-user CF prediction:                      {user_user_cf(R, user, item, k=3):.2f}")
print(f"Item-item CF prediction:                      {item_item_cf(R, user, item, k=3):.2f}")


## 9.4 Matrix factorization (the production-grade approach)

The big jump. Decompose the user-item rating matrix into low-rank factors:
$$R \approx P Q^\top$$
where $P \in \mathbb{R}^{n_u \times k}$ is the user factor matrix and $Q \in \mathbb{R}^{n_i \times k}$ is the item factor matrix. Each row $\mathbf{p}_u$ is user $u$'s preferences over $k$ latent dimensions ("how much they like action," "how much they like romance"); each row $\mathbf{q}_i$ is item $i$'s scoring on those same dimensions.

Predicted rating: $\hat{r}_{ui} = \mathbf{p}_u^\top \mathbf{q}_i$.

**Loss** (with regularization, computed only over observed ratings):
$$\mathcal{L} = \sum_{(u, i) \in \text{observed}} (r_{ui} - \mathbf{p}_u^\top \mathbf{q}_i)^2 + \lambda (\|P\|_F^2 + \|Q\|_F^2)$$

Solved via **SGD** (the practical approach) or **alternating least squares** (mathematically cleaner).

This is the algorithm that won the Netflix Prize (2009) and is the foundation of every modern recommender. Variants add biases, time effects, implicit feedback (BPR), and deep learning extensions.

In [ ]:
class MatrixFactorizationScratch:
    '''
    Matrix factorization via SGD with biases.

    Predicted rating:  r_hat(u, i) = mu + b_u + b_i + p_u^T q_i
    where mu = global mean, b_u = user bias, b_i = item bias, p_u and q_i are latent factors.

    Time complexity per epoch: O(|observed| * k)
    '''
    def __init__(self, n_factors=2, n_epochs=100, lr=0.01, reg=0.05, random_state=0):
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr = lr
        self.reg = reg
        self.random_state = random_state

    def fit(self, R):
        R = np.asarray(R, dtype=float)
        n_users, n_items = R.shape
        rng = np.random.default_rng(self.random_state)

        # Initialize factors small.
        self.P = rng.standard_normal((n_users, self.n_factors)) * 0.1
        self.Q = rng.standard_normal((n_items, self.n_factors)) * 0.1
        self.b_u = np.zeros(n_users)
        self.b_i = np.zeros(n_items)
        self.mu = np.nanmean(R)

        # Indices of observed ratings.
        obs = [(u, i, R[u, i]) for u in range(n_users) for i in range(n_items)
               if not np.isnan(R[u, i])]

        self.history_ = []
        for epoch in range(self.n_epochs):
            rng.shuffle(obs)
            total_loss = 0
            for u, i, r in obs:
                pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
                err = r - pred
                # SGD updates with regularization.
                self.b_u[u] += self.lr * (err - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (err - self.reg * self.b_i[i])
                P_u_old = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * P_u_old - self.reg * self.Q[i])
                total_loss += err ** 2
            self.history_.append(total_loss / len(obs))
        return self

    def predict(self, u, i):
        return self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]

    def predict_all(self):
        '''Return the full predicted rating matrix.'''
        n_users, n_items = self.P.shape[0], self.Q.shape[0]
        return self.mu + self.b_u[:, None] + self.b_i[None, :] + self.P @ self.Q.T


# Fit.
mf = MatrixFactorizationScratch(n_factors=2, n_epochs=300, lr=0.01, reg=0.05).fit(R)
R_pred = mf.predict_all()

# Compare with true ratings on the held-out (missing) entries.
mask = np.isnan(R) & ~np.isnan(true_ratings)
mse = ((R_pred[mask] - true_ratings[mask]) ** 2).mean()
print(f"MF training loss (final): {mf.history_[-1]:.4f}")
print(f"MF test MSE on held-out entries: {mse:.4f}")

# Plot training curve.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mf.history_)
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean squared error (observed)")
ax.set_title("Matrix factorization training curve")
plt.show()


## 9.5 Predicting top-k recommendations

In [ ]:
# For each user, find unrated items with the highest predicted rating.
def top_k_recommendations(R, R_pred, user_idx, k=3):
    unrated = np.isnan(R[user_idx])
    pred_scores = R_pred[user_idx].copy()
    pred_scores[~unrated] = -np.inf   # mask out items the user has rated
    top_idx = np.argsort(pred_scores)[-k:][::-1]
    return [(items[i], pred_scores[i]) for i in top_idx]


print("Top-3 recommendations per user (from items they haven't rated):\n")
for u in range(n_users):
    rec = top_k_recommendations(R, R_pred, u, k=3)
    print(f"  {users[u]}: {', '.join(f'{m}({s:.2f})' for m, s in rec)}")


## 9.6 The cold-start problem

The biggest practical limitation of CF: **new users and new items have no rating history**. CF methods fail completely on them.

| Strategy | When to use |
|---|---|
| **Popularity baseline** | Brand-new users — show most-popular items |
| **Demographic / feature-based** | New users with profile features (age, location, signup source) |
| **Content-based fallback** | New items with metadata (genre, tags, description) — recommend based on item similarity in feature space |
| **Hybrid (CF + content)** | Production systems: use content for new items and users, switch to CF as data accrues |
| **Active learning / onboarding** | Ask new users to rate a few items upfront (Netflix's "pick 3 movies" flow) |

**In 2026 production:** hybrid models are standard. CF is the core signal once you have data; content + demographics fill the cold-start gap.

## 9.7 Cheat table

In [ ]:
cf_cheat = pd.DataFrame([
    ["User-user CF",         "O(n_users² · d_avg)",  "O(n_users · d_avg)",
     "Simple, interpretable",  "Doesn't scale; user similarities unstable; sparse"],
    ["Item-item CF",         "O(n_items² · u_avg)",  "O(n_items · u_avg)",
     "Production default;  similarities stable",  "Cold start for new items"],
    ["Matrix factorization", "O(n_obs · k · epochs)", "O(k) per prediction",
     "Scales well; latent factors interpretable;  best accuracy", "Cold start; needs hyperparameter tuning"],
], columns=["Method", "Train time", "Predict time", "Strengths", "Weaknesses"])
cf_cheat


## 9.8 Interview Q&A

**Q: What's the difference between user-user and item-item collaborative filtering?**
> **User-user:** find users similar to the target user, recommend items they liked. Conceptually intuitive. Bad in production: user similarities are unstable (users change behavior over time), and user counts are usually much larger than item counts → similarity matrix is huge. **Item-item:** find items similar to the target item (similarity = which users rated both highly). Production default since Amazon's 2003 paper. Item similarities are more stable, and item counts are usually smaller.

**Q: What's matrix factorization, intuitively?**
> Decompose the rating matrix $R$ as $P Q^\top$ where $P$ holds **latent user preferences** over $k$ dimensions and $Q$ holds **latent item attributes** over those same dimensions. Dimensions often correspond loosely to interpretable concepts ("action movies," "romance movies," "indie films") — though the model discovers these without supervision. Predicted rating = dot product of user's preference vector with item's attribute vector. Won the Netflix Prize; foundation of every modern recommender.

**Q (trap): My collaborative filtering recommender gives identical recommendations to every user. What's wrong?**
> Probably one of: (a) **Popularity bias** — without mean-centering, popular items dominate; CF reduces to recommending whatever is most-rated. Subtract user/item means before computing similarities or use a model with explicit biases (mu + b_u + b_i + …). (b) **Too much regularization** — your factors collapse to zero. (c) **Cold-start:** if most users have very few ratings, predictions converge to global mean. Add explicit biases or fall back to popularity for low-rating users.

**Q: How do you handle the cold-start problem?**
> Multiple approaches: (1) **Popularity baseline** for brand-new users. (2) **Content-based filtering** — recommend items similar to ones the user explicitly rated based on item metadata (genre, tags). (3) **Hybrid models** — combine CF scores with content-based scores; weight depends on how much data you have for the user/item. (4) **Onboarding flow** — ask new users to rate ~5 items upfront. (5) **Demographic fallbacks** — use age/location/source as features. Production systems do (3) by default.

**Q (trap): The Netflix Prize winner was an ensemble of 100+ models. Why don't we use that in production?**
> Operational cost. Serving 100 model predictions per request is slow, expensive, and a nightmare to maintain. Companies use a single well-tuned matrix factorization (or a neural CF model) in production, accepting maybe 1-2% accuracy loss vs the ensemble. The Prize taught the field that **MF is the right framework**, not that 100-model ensembles are practical.

**Q: How is collaborative filtering different from content-based filtering?**
> **Collaborative:** uses only the user-item interaction matrix (ratings, clicks). Doesn't need item features. **Content-based:** uses item features (genre, tags, text descriptions) — recommends items similar to ones the user liked, based on those features. CF benefits from **shared user signal** (discovers preferences via "people who liked X also liked Y"); content-based handles cold-start better (works on new items if they have features). Production: hybrid is standard.

**Q (trap): I trained matrix factorization and the latent factors don't seem interpretable. Why?**
> Two reasons: (1) **Rotational invariance** — $PQ^\top = (PR)(R^{-1}Q^\top)^\top$ for any invertible $R$. The latent factors can be rotated arbitrarily without changing predictions, so they may not align with human concepts. (2) **No supervision** — there's no constraint that "dimension 1 should mean action movies." If interpretability matters, use **NMF (non-negative matrix factorization)** or add side-information constraints — these produce more interpretable factors at slight cost to predictive accuracy.


---
# 10. Isolation Forest — anomaly detection (bonus)

A different problem class. **Unsupervised outlier detection.** Anomalies are easier to **isolate** than normal points: a random split tree will reach an outlier in few splits, but a normal point requires many. Build many random trees, average the path length per point — short average path = anomaly.

## 10.1 The intuition

For each tree:
1. Pick a random feature.
2. Pick a random threshold within that feature's range.
3. Split. Recurse.
4. Stop when each point is isolated (or max depth reached).

**Anomalies tend to require fewer splits to isolate** (they're far from the bulk). Average path length across many trees → anomaly score. Short path = anomaly; long path = normal.

This is a beautiful inversion of standard tree-based reasoning: instead of growing trees to fit a label, we grow random trees and observe how quickly each point gets separated.

## 10.2 sklearn usage

In [ ]:
from sklearn.ensemble import IsolationForest

# Generate data with anomalies.
rng = np.random.default_rng(0)
X_normal = rng.normal(0, 1, size=(500, 2))
X_anomaly = rng.uniform(-6, 6, size=(20, 2))
X_iso = np.vstack([X_normal, X_anomaly])

iso = IsolationForest(n_estimators=100, contamination=0.04, random_state=0).fit(X_iso)
preds = iso.predict(X_iso)        # +1 = inlier, -1 = anomaly
scores = iso.decision_function(X_iso)  # higher = more "normal"

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_iso[preds == 1, 0],  X_iso[preds == 1, 1],
           c="steelblue", s=20, label="Inlier")
ax.scatter(X_iso[preds == -1, 0], X_iso[preds == -1, 1],
           c="tomato", s=80, marker="X", edgecolors="black", label="Anomaly")
ax.set_title("Isolation Forest — anomaly detection")
ax.legend()
plt.show()

print(f"Anomalies flagged: {(preds == -1).sum()}/{len(X_iso)} (expected ~20 true anomalies)")


## 10.3 Interview Q&A — anomaly detection

**Q: How does Isolation Forest detect anomalies?**
> Build random trees that recursively split the data on random features and thresholds. **Anomalies isolate quickly** (short path from root to leaf) because they're far from the bulk; normal points need more splits. Compute the average path length per point across many trees → anomaly score. The math: expected path length for normal points scales as $\log(n)$; for anomalies it's much shorter. Score $= 2^{-\text{avg path} / c(n)}$ where $c(n)$ is the expected path length for a binary search tree.

**Q: When would you use Isolation Forest over a one-class SVM?**
> Isolation Forest: faster (linear time), handles high dimensions, works on mixed feature types, no kernel choice. One-class SVM: better for very small datasets, more accurate boundary, but $O(n^2)$ training and needs feature scaling. In 2026 default: **Isolation Forest for tabular anomaly detection**, autoencoders for image/sequence data.

**Q (trap): What does the `contamination` parameter do, and how do I set it?**
> It's the **expected fraction of anomalies** in your training data. The threshold for "anomaly" is set so that exactly `contamination` fraction of training points are flagged. If you set it wrong, you'll mislabel. Best practice: if you have an estimate (say, 1% of credit card transactions are fraud), set `contamination=0.01`. If you don't know, set `contamination='auto'` (default in sklearn) — the algorithm uses a quantile-based heuristic.


---
# Closeout

## What you covered
- 4 clustering algorithms (K-Means, DBSCAN, Hierarchical, GMM)
- 3 dimensionality reduction methods (PCA, t-SNE, UMAP)
- TF-IDF for text representation
- Collaborative filtering (user-user, item-item, matrix factorization)
- Isolation Forest for anomaly detection

## The unsupervised cheat sheet — "when do I use what?"

| Problem signature | First reach for |
|---|---|
| Quick clustering, k known, spherical clusters | **K-Means** with k-means++ init |
| Non-spherical clusters, k unknown, want noise labels | **DBSCAN** (or HDBSCAN for varying density) |
| Small dataset, want to explore cluster hierarchy | **Agglomerative + dendrogram** |
| Need probabilistic (soft) cluster assignments | **GMM** with full covariance |
| Reduce dimensions for downstream modeling | **PCA** (linear, fast, transform-able) |
| Visualize high-dimensional clusters | **UMAP** (preferred) or **t-SNE** |
| Text → vectors for classification | **TF-IDF** + linear classifier baseline |
| User-item rating prediction at scale | **Matrix factorization** with SGD |
| Outlier detection in tabular data | **Isolation Forest** |

## When NOT to use these — common traps

- **K-Means on non-spherical clusters** → use GMM full cov or DBSCAN
- **PCA on fraud detection** → variance ≠ predictiveness; rare-class signal lives in low-variance directions
- **t-SNE features for downstream model** → t-SNE has no transform(); use UMAP instead
- **User-user CF in production with millions of users** → user similarities are unstable; use item-item or MF
- **CF for cold-start users** → add content-based fallback or onboarding flow
- **Isolation Forest with wrong `contamination`** → calibrate via held-out anomaly labels if possible

## What's next
**Notebook 3** is the **Evaluation Metrics Masterclass** — classification (binary, multi-class, multi-label), regression, ranking & recommendation, clustering. Multi-label depth and ROC/PR/calibration plots specifically called out as gaps in your reference docs.

Tell me when you want it.
